In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq
import json

from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))


class GroqLLM(DeepEvalBaseLLM):

    def load_model(self):
        return client

    def generate(self, prompt: str) -> str:
        response = client.chat.completions.create(
            model="qwen/qwen3.6-27b",   # or any Groq-supported model
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "llama-3.3-70b-versatile"


groq_model = GroqLLM()

faithfulness = FaithfulnessMetric(
    threshold=0.7,
    model=groq_model
)

test_case = LLMTestCase(
    input="Who is Gandhi?",
    actual_output=result["answer"],
    retrieval_context=[
        chunk["text"]
        for chunk in result["retrieved_chunks"]
    ]
)

faithfulness.measure(test_case)

print("Faithfulness :", faithfulness.score)
print("Passed       :", faithfulness.success)
print("Reason       :", faithfulness.reason)

In [ ]:
# ---------------------------
# Generate JSON Report
# ---------------------------
report = {
    "metadata": {
        "timestamp": datetime.now().isoformat(),
        "model": groq_model.get_model_name(),
        "metric": "Faithfulness",
        "threshold": faithfulness.threshold,
    },
    "test_case": {
        "input": test_case.input,
        "actual_output": test_case.actual_output,
        "retrieval_context": retrieval_context,
    },
    "evaluation": {
        "score": faithfulness.score,
        "passed": faithfulness.success,
        "reason": faithfulness.reason,
    }
}

with open("faithfulness_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

print("✅ JSON report saved as 'faithfulness_report.json'")